In [ ]:
import os

# 1. CONNECT TO GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

# 2. TELL COLAB WHERE YOUR KEY IS
# This points to the folder you just created in Phase 1
os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/My Drive/Kaggle"

# 3. DOWNLOAD KDD CUP 2015 DATA (AS A DATASET)
print("Downloading Data...")
# Use the -d flag for datasets instead of -c for competitions
!kaggle datasets download -d sst2023/kdd-cup-2015

# 4. UNZIP THE FILES
print("Unzipping...")
!unzip -q kdd-cup-2015.zip

# 5. LIST FILES (To confirm it worked)
print("Done! Here are your files:")
!ls

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/sst2023/kdd-cup-2015
License(s): unknown
100% 119M/119M [00:08<00:00, 15.6MB/s]

Unzipping...
Done! Here are your files:
date.csv  drive  kdd-cup-2015.zip  object.csv  sample_data  test  train


In [ ]:
import pandas as pd
import numpy as np
import torch
import os

print("Loading raw data...")
log_train = pd.read_csv("train/train/log_train.csv")
log_test = pd.read_csv("test/test/log_test.csv")
object_df = pd.read_csv("object.csv")
truth_train = pd.read_csv("train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv("test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

# ==========================================
# 1. FILTER FIRST 2 WEEKS
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

# ==========================================
# 2. MERGE OBJECT CATEGORY
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")


Loading raw data...


In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)


In [ ]:
# ==========================================
# 4. ENCODING
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

# Inverse vocabs -- needed to decode encoded ids back to strings inside each
# sliding window so FSLSM group membership (event/category name) can be checked.
inv_event_vocab = {i: v for v, i in event_vocab.items()}
inv_category_vocab = {i: v for v, i in category_vocab.items()}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat

    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)


In [ ]:
def compute_window_fslsm(evt_ids, cat_ids, gaps, inv_event_vocab, inv_category_vocab):
    """Compute the 4 FSLSM ratio scores for one sliding window.
    evt_ids, cat_ids, gaps are equal-length lists/arrays for the window (padding = evt_id 0).
    Returns [input_score, processing_score, perception_score, understanding_score].
    """
    visual = verbal = 0.0
    active = reflective = 0.0
    sensing = intuitive = 0.0
    sequential = global_ = 0.0

    for e_id, c_id, gap in zip(evt_ids, cat_ids, gaps):
        if e_id == 0:
            continue  # padding position, skip
        e = inv_event_vocab.get(e_id, "unknown")
        c = inv_category_vocab.get(c_id, "unknown")

        # Input: Visual vs Verbal
        if e == "video" or (e == "access" and c == "video"):
            visual += 1
        if e == "wiki" or e == "discussion" or (e == "access" and c == "html"):
            verbal += 1

        # Processing: Active vs Reflective
        if e == "problem" or e == "discussion":
            active += 1
        if e == "video" or (e == "access" and c == "html"):
            reflective += max(gap, 0.0)  # dwell-time proxy, already log1p-scaled

        # Perception: Sensing vs Intuitive
        if e == "problem":
            sensing += 1
        if e == "navigate" and c == "html":
            intuitive += 1

        # Understanding: Sequential vs Global
        if e == "navigate" and c in ("chapter", "sequential", "vertical"):
            sequential += 1
        if e == "page_close" or (e == "navigate" and c not in ("chapter", "sequential", "vertical")):
            global_ += 1

    def ratio(a, b):
        return (a - b) / (a + b) if (a + b) > 0 else 0.0

    return [
        ratio(visual, verbal),
        ratio(active, reflective),
        ratio(sensing, intuitive),
        ratio(sequential, global_),
    ]


In [ ]:
# ==========================================
# 4b. STATIC (LEARNER-LEVEL) FSLSM SCORES
# ==========================================
# One score per learner, computed from their FULL history — not per window.
# This matches "static learning-style feature" in the task and the aggregation
# pattern in Binary_Visual_Verbal_Scoring.pdf.

def compute_static_fslsm_for_df(df, inv_event_vocab, inv_category_vocab):
    scores = {}
    for _, row in df.iterrows():
        scores[row["student_id"]] = compute_window_fslsm(
            row["evt"], row["cat"], row["gaps"], inv_event_vocab, inv_category_vocab
        )
    return scores

train_static_fslsm = compute_static_fslsm_for_df(train_df, inv_event_vocab, inv_category_vocab)
test_static_fslsm  = compute_static_fslsm_for_df(test_df, inv_event_vocab, inv_category_vocab)

print(f"Computed static FSLSM scores for {len(train_static_fslsm)} train learners.")

Computed static FSLSM scores for 120542 train learners.


In [ ]:
# ==========================================
# 5. SLIDING WINDOWS (+ FSLSM SCORES)
# ==========================================

MAX_LEN, STRIDE = 150, 75

def build_windows(df, static_fslsm_dict):
    src_all, evt_all, cat_all, gap_all, y_all, ids_all, fslsm_all = [], [], [], [], [], [], []
    for _, row in df.iterrows():
        src, evt, cat, gaps = row["src"], row["evt"], row["cat"], row["gaps"]
        learner_score = static_fslsm_dict[row["student_id"]]  # <-- same vector for every window below

        for i in range(0, len(src), STRIDE):
            s, e, c, g = src[i:i+MAX_LEN], evt[i:i+MAX_LEN], cat[i:i+MAX_LEN], gaps[i:i+MAX_LEN]
            if not s:
                continue

            pad_len = MAX_LEN - len(s)
            s = s + [0]*pad_len; e = e + [0]*pad_len; c = c + [0]*pad_len; g = g + [0]*pad_len

            src_all.append(s); evt_all.append(e); cat_all.append(c); gap_all.append(g)
            fslsm_all.append(learner_score)   # <-- broadcast, not recomputed
            ids_all.append(row["student_id"])
            y_all.append(row["dropout"])

    return (np.array(src_all), np.array(evt_all), np.array(cat_all), np.array(gap_all),
            np.array(y_all), np.array(ids_all), np.array(fslsm_all, dtype=np.float32))

print("Building Windows (static FSLSM broadcast per learner)...")
src_train, evt_train, cat_train, gap_train, y_train, _, fslsm_train = build_windows(train_df, train_static_fslsm)
src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test = build_windows(test_df, test_static_fslsm)

print(f"Train windows: {src_train.shape[0]}, FSLSM score shape: {fslsm_train.shape}")
print("Mean FSLSM scores (train) [Input, Processing, Perception, Understanding]:")
print(fslsm_train.mean(axis=0))


Building Windows (static FSLSM broadcast per learner)...
Train windows: 168084, FSLSM score shape: (168084, 4)
Mean FSLSM scores (train) [Input, Processing, Perception, Understanding]:
[ 0.22033213  0.19460726  0.4996014  -0.99973226]


In [ ]:
# ==========================================
# 6. SAVE PROCESSED DATA
# ==========================================
os.makedirs("data/processed", exist_ok=True)
torch.save({
    "train": (src_train, evt_train, cat_train, gap_train, y_train, fslsm_train),
    "test": (src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test),
    "vocabs": (source_vocab, event_vocab, category_vocab),
    "truth_test": truth_test
}, "data/processed/processed_data_fslsm.pt")

print("Success! Saved to data/processed/processed_data_fslsm.pt")
print("Note: this file has the SAME src/evt/cat/gap/y/ids as the original processed_data.pt,")
print("plus one new 'fslsm' array per split -- so row 1 (baseline) and row 8 (with-FSLSM)")
print("in the comparison table are trained on identical windows/splits, differing only")
print("in whether the model consumes the extra 4 columns.")


Success! Saved to data/processed/processed_data_fslsm.pt
Note: this file has the SAME src/evt/cat/gap/y/ids as the original processed_data.pt,
plus one new 'fslsm' array per split -- so row 1 (baseline) and row 8 (with-FSLSM)
in the comparison table are trained on identical windows/splits, differing only
in whether the model consumes the extra 4 columns.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ==========================================
# 1. LOAD PROCESSED DATA (includes FSLSM scores)
# ==========================================
print("Loading processed data...")
data = torch.load("data/processed/processed_data_fslsm.pt", weights_only=False)

src_train, evt_train, cat_train, gap_train, y_train, fslsm_train = data["train"]
src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test = data["test"]
source_vocab, event_vocab, category_vocab = data["vocabs"]
truth_test = data["truth_test"]

print(f"Train windows: {src_train.shape[0]}, FSLSM feature dim: {fslsm_train.shape[1]}")


Loading processed data...
Train windows: 168084, FSLSM feature dim: 4


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, y, fslsm):
        self.src, self.evt, self.cat = torch.LongTensor(src), torch.LongTensor(evt), torch.LongTensor(cat)
        self.gaps, self.y = torch.FloatTensor(gaps), torch.FloatTensor(y)
        self.fslsm = torch.FloatTensor(fslsm)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.y[idx], self.fslsm[idx]

def make_loaders(batch_size=128):
    train_ds = SeqDataset(src_train, evt_train, cat_train, gap_train, y_train, fslsm_train)
    test_ds = SeqDataset(src_test, evt_test, cat_test, gap_test, y_test, fslsm_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               generator=torch.Generator().manual_seed(SEED))
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader


In [ ]:
# ==========================================
# 2. DEFINE MODEL (toggle: use_fslsm)
# ==========================================
class BiLSTMModel(nn.Module):
    def __init__(self, use_fslsm=False, fslsm_dim=4):
        super().__init__()
        self.use_fslsm = use_fslsm
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)

        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(128, 1)

        fc_in = 128 + (fslsm_dim if use_fslsm else 0)
        self.fc = nn.Linear(fc_in, 1)

    def forward(self, src, evt, cat, gaps, lengths, fslsm=None):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = torch.nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True)

        attn_scores = self.attention(out).squeeze(-1)
        attn_scores[~(src[:, :out.size(1)] != 0)] = -1e9  # Mask padding
        context = torch.sum(out * torch.softmax(attn_scores, dim=1).unsqueeze(-1), dim=1)

        if self.use_fslsm:
            context = torch.cat([context, fslsm], dim=1)

        return torch.sigmoid(self.fc(context)).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
# ==========================================
# 3. TRAIN + EVALUATE (one function, called for each config)
# ==========================================
def train_and_evaluate(use_fslsm, epochs=3, lr=0.001, batch_size=128):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # re-seed for fair comparison

    train_loader, test_loader = make_loaders(batch_size=batch_size)
    model = BiLSTMModel(use_fslsm=use_fslsm).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    tag = "WITH FSLSM" if use_fslsm else "BASELINE (no FSLSM)"
    print(f"\n=== Training {tag} ===")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, evt, cat, gaps, labels, fslsm in train_loader:
            src, evt, cat, gaps, labels, fslsm = [t.to(device) for t in (src, evt, cat, gaps, labels, fslsm)]

            optimizer.zero_grad()
            outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1), fslsm if use_fslsm else None)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    window_preds = []
    with torch.no_grad():
        for src, evt, cat, gaps, _, fslsm in test_loader:
            src, evt, cat, gaps, fslsm = [t.to(device) for t in (src, evt, cat, gaps, fslsm)]
            outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1), fslsm if use_fslsm else None)
            window_preds.extend(outputs.cpu().numpy())

    pred_df = pd.DataFrame({"student_id": test_ids, "pred": window_preds})
    eval_df = pred_df.groupby("student_id")["pred"].mean().reset_index().merge(truth_test, on="student_id")
    eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

    metrics = {
        "setting": tag,
        "accuracy": accuracy_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "precision": precision_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "recall": recall_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "f1": f1_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "auc": roc_auc_score(eval_df["dropout"], eval_df["pred"]),
    }
    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1 Score  : {metrics['f1']:.4f}")
    print(f"  AUC       : {metrics['auc']:.4f}")
    return metrics


In [ ]:
# ==========================================
# 4. RUN BOTH CONFIGS
# ==========================================
baseline_metrics = train_and_evaluate(use_fslsm=False)
fslsm_metrics = train_and_evaluate(use_fslsm=True)



=== Training BASELINE (no FSLSM) ===
  Epoch 1 Loss: 0.4554
  Epoch 2 Loss: 0.4414
  Epoch 3 Loss: 0.4381
  Accuracy  : 0.8436
  Precision : 0.8769
  Recall    : 0.9336
  F1 Score  : 0.9044
  AUC       : 0.8322

=== Training WITH FSLSM ===
  Epoch 1 Loss: 0.4557
  Epoch 2 Loss: 0.4415
  Epoch 3 Loss: 0.4380
  Accuracy  : 0.8441
  Precision : 0.8771
  Recall    : 0.9341
  F1 Score  : 0.9047
  AUC       : 0.8318


In [ ]:
# ==========================================
# 5. COMPARISON TABLE
# ==========================================
comparison = pd.DataFrame([baseline_metrics, fslsm_metrics]).set_index("setting")
comparison = comparison.round(4)
comparison.columns = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
print("\nComparison: Sequence-only vs Sequence + FSLSM scores")
comparison



Comparison: Sequence-only vs Sequence + FSLSM scores


,Accuracy,Precision,Recall,F1,AUC
setting,,,,,
BASELINE (no FSLSM),0.8436,0.8769,0.9336,0.9044,0.8322
WITH FSLSM,0.8441,0.8771,0.9341,0.9047,0.8318
